In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

DRIVE_REPO = Path('/content/drive/MyDrive/movie-recomender-search')
LOCAL_REPO = Path('/content/movie-recomender-search')

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

shutil.copytree(DRIVE_REPO, LOCAL_REPO)
os.chdir(LOCAL_REPO)

print('Working directory:', Path.cwd())
print('Drive repo:', DRIVE_REPO)
print('Local repo:', LOCAL_REPO)

# 04 - Model Comparison

Notebook tổng hợp metric từ `MF` và `NeuMF` để so sánh kết quả và lưu bảng/biểu đồ phục vụ báo cáo.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import FIGURES_DIR, METRICS_DIR
from AI.src.evaluation.compare import load_metric_files

print(f'Metrics dir: {METRICS_DIR}')
print(f'Figures dir: {FIGURES_DIR}')

In [ ]:
raw_metrics = load_metric_files(METRICS_DIR)
raw_metrics

In [ ]:
comparison = pd.DataFrame(
    {
        'model': raw_metrics['model'],
        'best_epoch': raw_metrics['best_epoch'],
        'val_rmse': raw_metrics['val_metrics'].apply(lambda x: x.get('rmse')),
        'val_mae': raw_metrics['val_metrics'].apply(lambda x: x.get('mae')),
        'test_rmse': raw_metrics['test_metrics'].apply(lambda x: None if x is None else x.get('rmse')),
        'test_mae': raw_metrics['test_metrics'].apply(lambda x: None if x is None else x.get('mae')),
        'train_rows': raw_metrics['train_rows'],
        'n_users': raw_metrics['n_users'],
        'n_items': raw_metrics['n_items'],
    }
)
comparison = comparison.sort_values('test_rmse', ascending=True, na_position='last').reset_index(drop=True)
comparison

In [ ]:
METRICS_DIR.mkdir(parents=True, exist_ok=True)
comparison_path = METRICS_DIR / 'model_comparison.csv'
comparison.to_csv(comparison_path, index=False)
print('Saved comparison table ->', comparison_path)

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURES_DIR / 'model_comparison.png'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison.plot(x='model', y='test_rmse', kind='bar', legend=False, ax=axes[0], color=['#4c72b0', '#dd8452'])
comparison.plot(x='model', y='test_mae', kind='bar', legend=False, ax=axes[1], color=['#4c72b0', '#dd8452'])
axes[0].set_title('Test RMSE')
axes[1].set_title('Test MAE')
axes[0].set_ylabel('Score')
axes[1].set_ylabel('Score')
plt.tight_layout()
plt.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved comparison figure ->', figure_path)

In [ ]:
winner = comparison.iloc[0]
print('Best model by test RMSE:')
print(winner)